# SmolLM2 instruction-tuning pilot — CPU

Run cells in order with the project `.venv` kernel. This notebook is unexecuted: the model and notebook dependencies were unavailable in the agent environment. It uses 7 reviewed training examples and 3 validation examples, never final test data.

Install kernel support from a terminal: `.\.venv\Scripts\python.exe -m pip install -r requirements-notebook.txt`. The first model load needs internet or a local Hugging Face model directory. This is supervised instruction tuning, not continued pretraining or reinforcement learning.

In [ ]:
from pathlib import Path
import sys, json, csv, html
root = Path.cwd()
if not (root / 'tokipona_torch').exists() and (root.parent / 'tokipona_torch').exists():
    root = root.parent
assert (root / 'tokipona_torch').exists(), 'Start from the project or notebooks directory'
import os
os.chdir(root)
sys.path.insert(0, str(root))
print('Project:', root)
print('Interpreter:', sys.executable)
import torch
from transformers import AutoTokenizer
from IPython.display import display, HTML
from tokipona_torch.data import read_rows, inspect_example
from tokipona_torch.model import load_base
from tokipona_torch.pilot import PilotConfig, run_pilot, generate_answer
config = PilotConfig()
# For offline use: config.model = r'C:\path\to\SmolLM2-360M-Instruct'
# Choose a new output directory for every run.
print(config)

## 1. Inspect the reviewed data

Read the question, desired answer, author and cited pages. The review was performed by the assistant; you can still correct the reviewed files. Changing the draft does not automatically update reviewed copies.

In [ ]:
train_rows = [row for _, row in read_rows(config.train)]
valid_rows = [row for _, row in read_rows(config.valid)]
print('Training:', len(train_rows), 'Validation:', len(valid_rows))
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2))

## 2. Render chat and inspect every token

SmolLM2 supplies its own chat template. The model sees the whole prompt, but only the final assistant answer and its ending are supervised. A label of `-100` excludes that token from the loss. At position *i*, the target is predicted by the logits at position *i−1*.

Padding is excluded by position, so a real end token remains supervised even when the padding and end token IDs are identical.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config.model, use_fast=True)
view = inspect_example(train_rows[0], tokenizer, config.max_length)
print(view['rendered_chat'])
print({k:v for k,v in view.items() if k not in ('tokens','rendered_chat','target_text')})
print('Supervised text:', repr(view['target_text']))
columns = ['position','token_id','piece','decoded','attention_mask','label','contributes_to_loss']
header = ''.join('<th>'+html.escape(c)+'</th>' for c in columns)
body = ''.join('<tr>'+''.join('<td>'+html.escape(repr(t[c]))+'</td>' for c in columns)+'</tr>' for t in view['tokens'])
display(HTML('<div style="max-height:420px;overflow:auto"><table><tr>'+header+'</tr>'+body+'</table></div>'))

In [ ]:
audit = [(row['id'], inspect_example(row, tokenizer, config.max_length)) for row in train_rows+valid_rows]
for example_id, item in audit:
    print(example_id, 'tokens:', item['retained_tokens'], 'targets:', item['supervised_tokens'], 'truncated:', item['truncated'], 'EOS supervised:', item['eos_supervised'])
assert all(not v['truncated'] and v['supervised_tokens'] and v['eos_supervised'] for _,v in audit)

## 3. Generate before training

Load the base in CPU float32. Fix the prompts and greedy generation settings before updating weights. These are development comparisons: training questions illustrate fitting; validation questions provide a limited check on transfer.

In [ ]:
torch.set_num_threads(config.threads)
from transformers import set_seed
set_seed(config.seed)
model, tokenizer, dtype = load_base(config.model, 'cpu')
print('Device:', next(model.parameters()).device, 'Dtype:', dtype)
dev_rows = train_rows[:2] + valid_rows[:2]
for row in dev_rows:
    print('\nQUESTION:', row['messages'][0]['content'])
    print('BASE:', generate_answer(model, tokenizer, row['messages'][:-1], config.max_new_tokens))

## 4. Run ten LoRA updates

Batch size is 1. Only q_proj/v_proj adapters in the final four layers are trainable. The runner repeats the fixed baseline comparison, trains, evaluates every five steps and saves results. Loss values are next-token prediction errors, not rewards. Expect the printed step log to show training loss, validation loss, supervised tokens, elapsed time and RAM.

If you change settings, choose a new `config.output`; existing runs are protected.

In [ ]:
manifest, comparisons = run_pilot(config, model_and_tokenizer=(model, tokenizer))
print('Adapter saved:', Path(config.output) / 'adapter')
print('Validation loss before:', manifest['initial_validation_loss'])
print('Validation loss after:', manifest['final_validation_loss'])

## 5. Inspect the loss history and compare answers

Training losses use different examples and need not fall at every step. With only three validation examples, avoid drawing broad conclusions. The generated answers may still be inaccurate; compare them with the source-supported references.

In [ ]:
with (Path(config.output) / 'loss.csv').open(encoding='utf-8') as f:
    history = list(csv.DictReader(f))
cols = list(history[0])
display(HTML('<table><tr>'+''.join('<th>'+html.escape(c)+'</th>' for c in cols)+'</tr>'+''.join('<tr>'+''.join('<td>'+html.escape(r[c])+'</td>' for c in cols)+'</tr>' for r in history)+'</table>'))

In [ ]:
for row in comparisons:
    display(HTML('<h3>'+html.escape(row['id']+' — '+row['split'])+'</h3><p>'+html.escape(row['prompt'][0]['content'])+'</p><table><tr><th>Before</th><th>After</th><th>Reference</th></tr><tr>'+''.join('<td style="vertical-align:top;max-width:360px">'+html.escape(row[k])+'</td>' for k in ('before','after','reference'))+'</tr></table>'))

Outputs are in the selected `runs/` directory: `loss.csv`, `token_inspection.json`, `before_after.json`, `run.json`, and `adapter/`. The run manifest records model revision when available, dependencies and dataset hashes. No final test data is loaded.